# Pillar B+ Colab Notebook

This notebook runs the final Pillar B+ architecture on Google Colab. It keeps the original Python files intact and calls them from notebook cells, so the saved adapter and evaluation files stay in Google Drive.

Use this notebook when the laptop has no CUDA GPU. Training and demo inference need the Phi-3.5-mini base model plus the LoRA adapter.

## 1. Select a GPU runtime

In Colab, use `Runtime -> Change runtime type -> GPU` before running the notebook. L4 or A100 is preferred because the training script was written for bf16. T4 may require the optional fp16 fallback cell below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

# Update this if your Drive folder has a different name.
CANDIDATE_PROJECT_DIRS = [
    Path('/content/drive/MyDrive/BeyondFlesch_PillarBPlus/Final_Architecture-Pillar_B plus'),
    Path('/content/drive/MyDrive/Beyond-Flesch/Beyond-Flesch/Final_Architecture-Pillar_B plus'),
    Path('/content/drive/MyDrive/Final_Architecture-Pillar_B plus'),
]

PROJECT_DIR = next((p for p in CANDIDATE_PROJECT_DIRS if p.exists()), CANDIDATE_PROJECT_DIRS[0])
print('Using project dir:', PROJECT_DIR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        'Could not find the Pillar B+ folder on Drive. Upload the project folder, '
        'then update PROJECT_DIR in this cell.'
    )

os.chdir(PROJECT_DIR)
print('Current working directory:', Path.cwd())

## 2. Install dependencies

Colab already includes PyTorch. This installs the project-specific packages used by Pillar B+ and the Gradio demo.

In [ ]:
!pip install -q transformers peft accelerate gradio codecarbon textstat sentence-transformers einops scikit-learn pandas numpy huggingface_hub

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab, switch Runtime type to GPU.')

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA version:', torch.version.cuda)
print('bf16 supported:', torch.cuda.is_bf16_supported())

if not torch.cuda.is_bf16_supported():
    print('\nWarning: this GPU may not support bf16. Use L4/A100 if possible, or run the optional fp16 fallback cell.')

In [ ]:
# Optional: run this only if Hugging Face asks for authentication.
# from huggingface_hub import notebook_login
# notebook_login()

## 3. Check project files

Demo inference needs `demo/`, `scripts/`, and the saved LoRA adapter. Training/evaluation also needs the original `generalization_final/outputs` split and judge files beside this folder.

In [ ]:
from pathlib import Path

required_files = [
    'scripts/pillar_b_plus.py',
    'scripts/pillar_b_tiny_teacher.py',
    'scripts/evaluate_all.py',
    'demo/app.py',
    'data/adv_concept.csv',
]

for rel in required_files:
    path = PROJECT_DIR / rel
    print(('OK      ' if path.exists() else 'MISSING '), rel)

REPO_ROOT = PROJECT_DIR.parent
GF_OUT = REPO_ROOT / 'generalization_final' / 'outputs'
print('\nExpected training data root:', GF_OUT)
print('Training data root exists:', GF_OUT.exists())

if not GF_OUT.exists():
    print('\nTraining will fail until the generalization_final/outputs folder is copied to Drive next to Final_Architecture-Pillar_B plus.')

## 4. Optional fp16 fallback for T4

Only use this if Colab gives you a T4 or another GPU where `bf16 supported` printed `False`. The original project was trained with bf16, so L4/A100 is the cleaner choice.

In [ ]:
USE_FP16_FALLBACK = False

if USE_FP16_FALLBACK:
    patch_files = [
        PROJECT_DIR / 'scripts' / 'pillar_b_plus.py',
        PROJECT_DIR / 'demo' / 'app.py',
    ]
    for path in patch_files:
        text = path.read_text()
        text = text.replace('dtype=torch.bfloat16', 'torch_dtype=torch.float16')
        text = text.replace('dtype=dtype,', 'torch_dtype=dtype,')
        text = text.replace('return torch.device("cuda"), torch.bfloat16', 'return torch.device("cuda"), torch.float16')
        path.write_text(text)
        print('Patched for fp16:', path)
else:
    print('Leaving source files unchanged.')

## 5. Train Pillar B+

This calls the original `scripts/pillar_b_plus.py`. Because the working directory is on Drive, the adapter and eval reports are saved on Drive too.

Expected adapter output: `outputs/models/pillar_b_plus__phi-3.5-mini-instruct__lora/best/`

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    !python scripts/pillar_b_plus.py --epochs 2 --target-per-class 3500 --lora-r 32
else:
    print('Set RUN_TRAINING = True when the data folders are present and you are ready to train.')

## 6. Verify the saved adapter

Run this after training, or after copying a previously trained adapter into the expected Drive folder.

In [ ]:
ADAPTER_DIR = PROJECT_DIR / 'outputs' / 'models' / 'pillar_b_plus__phi-3.5-mini-instruct__lora' / 'best'
print('Adapter dir:', ADAPTER_DIR)
print('Exists:', ADAPTER_DIR.exists())

if ADAPTER_DIR.exists():
    for item in sorted(ADAPTER_DIR.iterdir()):
        print(item.name)
else:
    print('No adapter found yet. Train first or copy the saved best/ adapter folder here.')

## 7. Load the model for inference

This loads `microsoft/Phi-3.5-mini-instruct`, attaches the saved LoRA adapter, and defines a `classify()` function.

In [ ]:
import sys
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from pillar_b_tiny_teacher import build_prompt, _find_letter_token_ids

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f'LoRA adapter not found: {ADAPTER_DIR}')

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'
DEVICE = torch.device('cuda')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    attn_implementation='sdpa',
).to(DEVICE)

model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR)).to(DEVICE).eval()
letter_ids = _find_letter_token_ids(tokenizer)
levels = ['elementary', 'middle', 'high']

@torch.no_grad()
def classify(text):
    prompt = build_prompt(text)
    encoded = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1280).to(DEVICE)
    start = time.perf_counter()
    output = model(**encoded, use_cache=False)
    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - start) * 1000

    last = output.logits[0, -1, :]
    logits = torch.stack([last[letter_ids['E']], last[letter_ids['M']], last[letter_ids['H']]])
    probs = torch.softmax(logits.float(), dim=-1).detach().cpu().tolist()
    pred_idx = max(range(3), key=lambda i: probs[i])
    return {
        'prediction': levels[pred_idx],
        'probabilities': dict(zip(levels, probs)),
        'latency_ms': latency_ms,
    }

print('Model loaded. Letter IDs:', letter_ids)

In [ ]:
classify('Photosynthesis uses light energy to convert carbon dioxide and water into glucose and oxygen.')

## 8. Launch a Gradio demo

This creates a Colab-shareable Gradio link without editing `demo/app.py`.

In [ ]:
import gradio as gr

def classify_for_ui(text):
    text = (text or '').strip()
    if not text:
        return {}, 'Paste text to classify.', ''
    result = classify(text)
    probs = result['probabilities']
    pred = result['prediction']
    detail = (
        f'### Prediction: **{pred}**\n\n'
        f"| level | probability |\n| --- | --- |\n"
        f"| elementary | {probs['elementary'] * 100:.1f}% |\n"
        f"| middle | {probs['middle'] * 100:.1f}% |\n"
        f"| high | {probs['high'] * 100:.1f}% |\n\n"
        f"**Latency:** {result['latency_ms']:.1f} ms"
    )
    return probs, detail, f"{result['latency_ms']:.1f} ms"

examples = [
    ['What is mitosis?'],
    ['Why is the sky blue?'],
    ['Solve for x: 3x + 7 = 22.'],
    ['Photosynthesis involves light-dependent reactions in the thylakoid membrane and the Calvin cycle in the stroma.'],
]

with gr.Blocks(title='Pillar B+ Text-Difficulty Classifier') as demo:
    gr.Markdown('# Pillar B+ Text-Difficulty Classifier')
    with gr.Row():
        with gr.Column():
            text_box = gr.Textbox(lines=8, label='Paste text to classify')
            button = gr.Button('Classify', variant='primary')
            gr.Examples(examples=examples, inputs=[text_box])
        with gr.Column():
            label_output = gr.Label(label='Class probabilities', num_top_classes=3)
            detail_output = gr.Markdown()
            latency_output = gr.Textbox(label='Latency', interactive=False)
    button.click(classify_for_ui, inputs=[text_box], outputs=[label_output, detail_output, latency_output])

demo.launch(share=True, debug=True)

## 9. What gets saved

Because `PROJECT_DIR` is inside Google Drive, these artifacts remain saved after Colab disconnects:

- `outputs/models/pillar_b_plus__phi-3.5-mini-instruct__lora/best/`
- `outputs/eval_reports/pillar_b_plus__phi-3.5-mini-instruct.eval.json`
- `outputs/emissions/emissions.csv`

The full Phi-3.5 base model is downloaded into the Colab/Hugging Face cache each session unless you separately configure a Drive cache.